In [29]:
from danial import model, dataloader, loss_h
import torch.optim as optim
import torch

In [2]:
mod = model.Model()
print(mod)

/Users/dania/code/fyp/MHNet/vengeance/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/dania/code/fyp/MHNet/vengeance/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model(
  (backbone): VGGFeatureExtractor(
    (block1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (block2): Sequential(
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (block3): Sequential(
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3,

In [3]:
mod.train()
criterion = loss_h.HungarianMatcher(
        num_classes=69,
        matcher_cost_class=1,
        matcher_cost_bbox=5,
        matcher_cost_giou=2,
        loss_ce=1,
        loss_bbox=5,
        loss_giou=2,
        eos_coef=0.1
    )

In [4]:
test = dataloader.load_image("test-images/camo-fish.png", target_size=(224, 224))
# label = 

In [5]:
out = mod(test)
print(out["pred_logits"].shape)
print(out["pred_boxes"].shape)
print(out["recovered_features"].shape)
print(out["mask_output"].shape)

torch.Size([1, 100, 6])
torch.Size([1, 100, 4])
torch.Size([1, 64, 224, 224])
torch.Size([1, 1, 224, 224])


In [6]:
def coco_to_detr_targets(coco_annotations, img_width, img_height):
    """
    Convert COCO format to DETR target format
    Args:
        coco_annotations: List of annotation dicts for ONE image
        img_width, img_height: Image dimensions
    Returns:
        dict with 'labels' and 'boxes' tensors
    """
    labels = []
    boxes = []
    
    # COCO format: [x_min, y_min, width, height] in pixels
    x_min, y_min, w, h = coco_annotations["annotations"][0]['bbox']
    
    # Convert to normalized [cx, cy, w, h]
    cx = (x_min + w / 2) / img_width
    cy = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    
    # Category ID (0-indexed, COCO is 1-indexed)
    class_id = coco_annotations["annotations"][0]['category_id'] - 1  # Convert to 0-indexed
    
    labels.append(class_id)
    boxes.append([cx, cy, w_norm, h_norm])
    
    return {
        'labels': torch.tensor(labels, dtype=torch.long),  # [num_objects]
        'boxes': torch.tensor(boxes, dtype=torch.float32)   # [num_objects, 4]
    }

In [7]:
import json

with open("annotation.json", "r") as f:
    annotations = json.load(f)
    width, height = annotations["images"][0]["width"], annotations["images"][0]["height"]

targets = coco_to_detr_targets(annotations, width, height)
# print(annotations)
# for ann in annotations:
#     print(ann)

In [8]:
from danial import test_hungarian

In [9]:
batch_size = 1
num_queries = 100
num_classes = 5

criterion = loss_h.HungarianMatcher(
    num_classes=num_classes,
    matcher_cost_class=1,
    matcher_cost_bbox=5,
    matcher_cost_giou=2,
    loss_ce=1,
    loss_bbox=5,
    loss_giou=2,
    eos_coef=0.1
)

In [10]:
mod_out = {
    'pred_logits': out["pred_logits"],
    'pred_boxes': out["pred_boxes"]
}
print(mod_out["pred_logits"].shape)
print(mod_out["pred_boxes"].shape)

torch.Size([1, 100, 6])
torch.Size([1, 100, 4])


In [ ]:
targets = []
for i in range(batch_size):
    # Use 1 ground-truth object (from annotations)
    # Build the GT box (COCO format is [x_min, y_min, width, height] in pixels)
    GT_box = torch.tensor(annotations["annotations"][0]["bbox"]).to(torch.float32).unsqueeze(0)
    # Extract class id from annotations and convert to 0-indexed long tensor
    class_id = annotations["annotations"][0]["category_id"] - 1
    labels = torch.tensor([class_id], dtype=torch.int64)
    targets.append({'labels': labels, 'boxes': GT_box})

print(labels)
print(labels.shape)
print(labels.dtype)
print(GT_box)
print(GT_box.shape)
print(GT_box.dtype)

tensor([0])
torch.Size([1])
torch.int64
tensor([[112., 203., 552., 239.]])
torch.Size([1, 4])
torch.float32


In [21]:
loss = criterion(mod_out, targets)

In [23]:
print(loss['loss_total'])

tensor(5524.2646, grad_fn=<AddBackward0>)


In [32]:
optimizer = optim.AdamW(mod.parameters(), lr=1e-5)
optimizer.zero_grad()

In [34]:
loss['loss_total'].backward()

/Users/dania/code/fyp/MHNet/vengeance/lib/python3.10/site-packages/torch/autograd/__init__.py:266: UserWarning: Skipping device NVIDIA GeForce GT 750M that does not support Metal 2.0 (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSDevice.mm:101.)
  Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [38]:
from torchviz import make_dot
make_dot(loss['loss_total'], params=dict(list(mod.named_parameters()))).render("loss_graph.jpg", format="png")

'loss_graph.jpg.png'

In [40]:
print(mod)

Model(
  (backbone): VGGFeatureExtractor(
    (block1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (block2): Sequential(
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (block3): Sequential(
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3,